In [1]:
!pip install pandas numpy matplotlib seaborn scikit-learn

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: C:\Python314\python.exe -m pip install --upgrade pip


In [ ]:
# 1. Import Libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_classification
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)


# 2. Generate Synthetic Adult Census Dataset

X, y = make_classification(
    n_samples=500,
    n_features=7,
    n_informative=5,
    n_redundant=0,
    n_classes=2,
    weights=[0.7, 0.3],
    random_state=42
)


# 3. Convert Features into Meaningful Ranges

scaler = MinMaxScaler()

X_scaled = scaler.fit_transform(X)

df = pd.DataFrame({
    "Age": (X_scaled[:, 0] * 47 + 18).astype(int),
    "Education_Num": (X_scaled[:, 1] * 12 + 1).astype(int),
    "Capital_Gain": (X_scaled[:, 2] * 10000).astype(int),
    "Capital_Loss": (X_scaled[:, 3] * 2000).astype(int),
    "Hours_Per_Week": (X_scaled[:, 4] * 50 + 10).astype(int),
    "Work_Experience": (X_scaled[:, 5] * 40).astype(int),
    "Family_Members": (X_scaled[:, 6] * 8).astype(int)
})

df["Income"] = y

# 0 = <=50K, 1 = >50K

print("Generated Adult Census Dataset:")
print(df.head())


# 4. Save Dataset

df.to_csv("adult_census.csv", index=False)

print("\nDataset saved successfully!")


# 5. Explore Dataset

print("\nDataset Shape:")
print(df.shape)

print("\nDataset Information:")
df.info()

print("\nStatistical Summary:")
print(df.describe())

print("\nMissing Values:")
print(df.isnull().sum())

print("\nIncome Distribution:")
print(df["Income"].value_counts())


# 6. Visualize Income Distribution

sns.countplot(x="Income", data=df)

plt.title("Income Distribution")
plt.xlabel("Income (0 = <=50K, 1 = >50K)")
plt.ylabel("Number of People")
plt.show()


# 7. Correlation Analysis

correlation = df.corr()

plt.figure(figsize=(9, 6))

sns.heatmap(
    correlation,
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Correlation Heatmap")
plt.show()


# 8. Separate Features and Target

X = df.drop("Income", axis=1)
y = df["Income"]


# 9. Split Dataset into Training and Testing Sets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining Data:", X_train.shape)
print("Testing Data:", X_test.shape)


# 10. Create Decision Tree Classifier

model = DecisionTreeClassifier(
    criterion="entropy",
    max_depth=5,
    random_state=42
)


# 11. Train the Model

model.fit(X_train, y_train)

print("\nDecision Tree Model Training Completed!")


# 12. Predict Income

y_pred = model.predict(X_test)


# 13. Compare Actual and Predicted Values

comparison = pd.DataFrame({
    "Actual Income": y_test.values,
    "Predicted Income": y_pred
})

print("\nActual vs Predicted:")
print(comparison.head(10))


# 14. Evaluate Model Performance

accuracy = accuracy_score(y_test, y_pred)

precision = precision_score(y_test, y_pred)

recall = recall_score(y_test, y_pred)

f1 = f1_score(y_test, y_pred)

print("\nModel Performance:")
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)


# 15. Confusion Matrix

cm = confusion_matrix(y_test, y_pred)

print("\nConfusion Matrix:")
print(cm)

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["<=50K", ">50K"],
    yticklabels=["<=50K", ">50K"]
)

plt.xlabel("Predicted Income")
plt.ylabel("Actual Income")
plt.title("Confusion Matrix")
plt.show()


# 16. Classification Report

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["<=50K", ">50K"]
))


# 17. Feature Importance

importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print("\nFeature Importance:")
print(importance)

plt.figure(figsize=(9, 5))

sns.barplot(
    x="Importance",
    y="Feature",
    data=importance
)

plt.title("Feature Importance")
plt.show()


# 18. Visualize Decision Tree

plt.figure(figsize=(18, 10))

plot_tree(
    model,
    feature_names=X.columns,
    class_names=["<=50K", ">50K"],
    filled=True,
    rounded=True,
    fontsize=8
)

plt.title("Decision Tree Classifier")
plt.show()


# 19. Predict Income for a New Person

new_person = pd.DataFrame({
    "Age": [35],
    "Education_Num": [12],
    "Capital_Gain": [5000],
    "Capital_Loss": [0],
    "Hours_Per_Week": [40],
    "Work_Experience": [15],
    "Family_Members": [3]
})

prediction = model.predict(new_person)

if prediction[0] == 1:
    print("\nPredicted Income: >50K")
else:
    print("\nPredicted Income: <=50K")